# Medical imaging

## Practicum 1: Computed Tomography Image Visualisation & Reconstruction
Oct-Nov 2026
==============================================================================================

In this practicum, you will work with several topics covered in the theory sessions (Block 1). In particular you will (1) simulate a simple and more complex phantoms, (2) apply Radom forward transformation to simulate the computed tomography (CT) projections over a single slice (i.e., sinograms), and (3) reconstruct the original phantom intensity values using different image reconstruction algorithms. In addition, the Shepp–Logan (head) phantom will be used to analyse noise during reconstruction.

### Aims:
- Generate simple and complex phantoms.
- Understand the principles of CT image reconstruction using a fan beam geometry with the aid of ``skimage``.
- Be able to generate and understand image projections (i.e., sinograms).
- Reconstruct slices from simple and more complex objects (i.e., Shepp–Logan phantom) using back projection.
- Analyse and compare reconstructed data with ground truth data.
- Observe the difference between ideal (noise free) and noisy image reconstruction.

``Remember to comment your code and provide some discussion on the results obtained for each section.``



In [1]:
%matplotlib inline

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon

In [3]:
!pip install pydicom --quiet

In [4]:
from google.colab import drive

try:
  drive.mount('/content/drive')
except Exception as e:
  print(e)

Error: credential propagation was unsuccessful


### 1.1 Visualise a lung CT dicom image and create a phantom (2 points)

Open and visualise the Lung CT volume stored at the `LungCT-Diagnosis` accompanied folder. You can use mango viewer, ITK-Snap, etc. Then, using python commands, read each of the indivdual `dcm` images, and display some 2D slices. Some basic image processing might be needed to adjust the contrast/windows level for better visualisation of all tissues. Then, create a 3D phantom with the ribcages based on their HU (e.g., thresholding) and visualise the 3D rendering. Congratulations, you have created your first anthropomorphic phantom!

<img src='https://github.com/sacumesh/medical-imagining-labs/blob/sachiththa/P1/images_4_notebook/LungPhantomImage.png?raw=1' width="400">


<img src='https://github.com/sacumesh/medical-imagining-labs/blob/sachiththa/P1/images_4_notebook/LungPhantomRendering.png?raw=1' width="400">


Data was taken from https://www.cancerimagingarchive.net/collection/lungct-diagnosis/


In [5]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pydicom as pdc
from pydicom.filereader import dcmread
from skimage.exposure import rescale_intensity
from skimage.morphology import ball, binary_opening, binary_closing, remove_small_objects
from skimage.measure import marching_cubes
from scipy.ndimage import gaussian_filter
import plotly.graph_objects as go

In [6]:
# ---------- 1) Read DICOM series ----------
DATA_DIR = Path("/content/drive/MyDrive/Medical Imaging/P1-student/P1-student/LungCT-Diagnosis")  # set to your dataset root

def read_series(dicom_dir: Path):
    # Find and sort all DICOM files recursively in the directory
    files = sorted([p for p in dicom_dir.rglob("*.dcm")])
    assert files, f"No DICOMs under {dicom_dir}"

    # Read headers to get the Z position and instance number for sorting
    metas = []
    for f in files:
        ds = dcmread(str(f), stop_before_pixels=True, force=True)
        z = None
        if hasattr(ds, "ImagePositionPatient"):
            z = float(ds.ImagePositionPatient[2])
        inst = int(getattr(ds, "InstanceNumber", 0))
        metas.append((f, z, inst))

    # Sort slices by patient position if available, otherwise by instance number
    if all(m[1] is not None for m in metas):
        metas.sort(key=lambda x: x[1])
    else:
        metas.sort(key=lambda x: x[2])

    # Read pixel arrays and convert raw values to Hounsfield Units (HU)
    slices, zs = [], []
    for f, z, _ in metas:
        ds = dcmread(f, stop_before_pixels=False, force=True)
        slope = float(getattr(ds, "RescaleSlope", 1))
        intercept = float(getattr(ds, "RescaleIntercept", 0))
        hu = ds.pixel_array.astype(np.float32) * slope + intercept
        slices.append(hu)
        zs.append(z)

    # Get slice spacing in X and Y directions
    dy, dx = map(float, ds.PixelSpacing)

    # Calculate slice spacing in Z direction using median difference, or use slice thickness
    if len(zs) > 1 and zs[0] is not None:
        dz = float(np.median(np.diff(zs)))
    else:
        dz = float(getattr(ds, "SliceThickness", 1.0))

    # Pack spacings and stack slices to form a 3D volume array
    spacing = (dz, dy, dx)
    vol_hu = np.stack(slices).astype(np.float32)

    return vol_hu, spacing

In [7]:
# ---------- 2) 2D visualisation with window/level ----------

# Read every slice (already sorted by z) into a list
vol_hu, spacing = read_series(DATA_DIR)

print("volume shape (z, y, x):", vol_hu.shape)
print("voxel spacing (mm):", spacing)
print(f"HU range: {vol_hu.min():.0f} .. {vol_hu.max():.0f}   (air = -1000, soft tissue = 0..100, bone > 300)")

# %% ---------- 2) 2D visualisation with window/level (Static Grid) ----------
windows = {"lung": (-600, 1500), "soft tissue": (40, 400), "bone": (400, 1800)}

# Select a few slices across the volume (e.g., at 25%, 50%, and 75% depth)
num_slices = vol_hu.shape[0]
selected_slices = [num_slices // 4, num_slices // 2, (3 * num_slices) // 4]

# Plot a grid of slices and window presets
fig, axes = plt.subplots(len(windows), len(selected_slices), figsize=(16, 14))

for r_idx, (win_name, (c, w)) in enumerate(windows.items()):
    # Calculate physical display range limits from center and width
    vmin = c - w / 2
    vmax = c + w / 2
    for c_idx, z in enumerate(selected_slices):
        ax = axes[r_idx, c_idx]
        im = ax.imshow(vol_hu[z], cmap="gray", vmin=vmin, vmax=vmax)
        ax.set_title(f"Slice {z} ({win_name} window)")
        ax.axis("on")  # Display the axes
        fig.colorbar(im, ax=ax, label="HU Scale (Windowed)")  # Display the colorbar

plt.tight_layout()
plt.show()

AssertionError: No DICOMs under /content/drive/MyDrive/Medical Imaging/P1-student/P1-student/LungCT-Diagnosis

In [ ]:
from scipy.ndimage import gaussian_filter, binary_fill_holes, binary_closing, label
from skimage.filters import apply_hysteresis_threshold
from skimage.morphology import ball, remove_small_objects

# Set threshold limits for identifying bone voxels
BONE_TH  = 250     # HU - lowest limit to be considered bone if connected to seed
SEED_TH  = 400     # HU - high limit representing definite cortical bone
MIN_VOL  = 1000    # mm3 - minimum size to keep (removes small noise)
MAX_HOLE = 150     # mm2 - maximum hole size to fill (like marrow cavities)

# Step 1: Smooth the volume slightly to reduce image noise
smoothed_vol = gaussian_filter(vol_hu, sigma=0.7)
dx, dy, dz = spacing
voxel_mm3 = dx * dy * dz

# Step 2: Create a body mask to isolate the patient and remove scanner bed/cables
body_mask = smoothed_vol > -500
lbl, n = label(body_mask)
sizes = np.bincount(lbl.ravel())
sizes[0] = 0                               # ignore background label
body_mask = lbl == sizes.argmax()          # keep only the largest object (the patient)
for z in range(body_mask.shape[0]):        # fill internal lung regions so they stay inside the body mask
    body_mask[z] = binary_fill_holes(body_mask[z])

# Step 3: Apply hysteresis (dual) thresholding only inside the body mask
# This detects true bone structure while ignoring isolated high-intensity noise
vol_body  = np.where(body_mask, smoothed_vol, -1000.0)
bone_mask = apply_hysteresis_threshold(vol_body, low=BONE_TH, high=SEED_TH)

# Step 4: Close small gaps in the bone mask using morphological closing
bone_mask = binary_closing(bone_mask, structure=ball(1))

# Step 5: Fill small 2D holes (such as bone marrow) slice-by-slice
max_hole_px = int(MAX_HOLE / (dy * dx))
for z in range(bone_mask.shape[0]):
    holes = binary_fill_holes(bone_mask[z]) & ~bone_mask[z]
    hl, nh = label(holes)
    if nh > 0:
        areas = np.bincount(hl.ravel())
        small = np.flatnonzero(areas <= max_hole_px)
        small = small[small > 0]           # ignore zero label
        bone_mask[z] |= np.isin(hl, small) # fill only the small holes

# Step 6: Remove very small 3D objects that don't meet the volume threshold
bone_mask = remove_small_objects(bone_mask, min_size=int(MIN_VOL / voxel_mm3))

# Compare voxel counts between simple thresholding and our cleaned bone mask
naive_mask = vol_hu >= BONE_TH
for name, m in [("naive threshold", naive_mask), ("final bone mask", bone_mask)]:
    _, n = label(m)
    print(f"{name:16s}: {m.sum():>9,} voxels, {n:5d} objects, {m.sum() * voxel_mm3 / 1000:.0f} cm3")

# Visualize the comparison on selected slices
num_slices = vol_hu.shape[0]
selected_slices = [num_slices // 4, num_slices // 2, (3 * num_slices) // 4]

fig, axes = plt.subplots(len(selected_slices), 2, figsize=(12, 5 * len(selected_slices)))
for idx, z in enumerate(selected_slices):
    # Left side: Simple thresholding overlay (yellow contour)
    axes[idx, 0].imshow(vol_hu[z], cmap="gray")
    axes[idx, 0].contour(naive_mask[z], levels=[0.5], colors="y", linewidths=0.7)
    axes[idx, 0].set_title(f"Slice {z} - naive >= {BONE_TH} HU")
    axes[idx, 0].axis("off")

    # Right side: Advanced bone mask overlay (red contour)
    axes[idx, 1].imshow(window(vol_hu[z], 400, 1800), cmap="gray")
    axes[idx, 1].contour(bone_mask[z], levels=[0.5], colors="r", linewidths=0.7)
    axes[idx, 1].set_title(f"Slice {z} - final bone mask")
    axes[idx, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter
from skimage.measure import marching_cubes
import plotly.graph_objects as go

field = gaussian_filter(np.pad(bone_mask, 1).astype(np.float32), sigma=(0.5, 1.0, 1.0))

verts, faces, normals, values = marching_cubes(field, level=0.5, spacing=spacing, step_size=1)
print(f"mesh: {len(verts):,} vertices, {len(faces):,} faces")

# verts are in (z, y, x) order, in mm
z, y, x = verts.T

mesh = go.Mesh3d(
    x=x, y=y, z=z,
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    color="lightgray",
    lighting=dict(ambient=0.45, diffuse=0.8, specular=0.15),
)
fig = go.Figure(mesh)
fig.update_layout(
    scene=dict(xaxis_title="X (mm)", yaxis_title="Y (mm)", zaxis_title="Z (mm)",
               aspectmode="data"),           # keep the true proportions
    width=800, height=800, margin=dict(l=0, r=0, t=30, b=0),
    title="3D ribcage phantom",
)
fig.show()

#### Discussion 1.1 – What, how and why

**What we did**
1. We read all the `.dcm` files, sorted them by their position along the body, and stacked them into one 3D volume of **68 × 512 × 512 voxels** (voxel size **5.0 × 0.83 × 0.83 mm**).
2. We changed the stored numbers into **Hounsfield Units (HU)** with `HU = pixel × RescaleSlope + RescaleIntercept`. The values go from −3024 to 3071 HU.
3. We showed 3 slices with 3 display windows (lung, soft tissue, bone).
4. We found the bone with a threshold on the HU values, cleaned the mask, and made a 3D surface with marching cubes. This is our ribcage phantom.

**Why HU?** The raw numbers in a DICOM file are different for every scanner. The DICOM standard gives a simple rule (Rescale Slope and Rescale Intercept) to change them into HU [1]. In HU, water = 0, air = −1000, soft tissue is about −100 to +100, and dense bone is above 1000 [2]. Only after this step can we say "bone = everything above 300 HU".

**Why windowing?** A CT image has about 4000 different HU values, but a screen can only show about 256 grey levels. The window chooses which HU values we look at. The *level* sets how bright the image is. The *width* sets how strong the contrast is [3]. We used the usual values: lung −600 / 1500, soft tissue 40 / 400, bone 400 / 1800 [4].

**Why not a simple threshold?** The rule `HU >= 250` picks everything that is bright: the table, cables, noise and vessels. It gave **322 133 voxels in 7 839 separate pieces (1110 cm³)**. Our cleaned method (body mask → hysteresis threshold 250 / 400 HU → closing → fill small holes → remove pieces smaller than 1000 mm³) gave **251 714 voxels in only 7 pieces (867 cm³)**. The key trick is the hysteresis threshold: a voxel counts as bone only if it is ≥ 250 HU **and** it is connected to a "sure" bone voxel ≥ 400 HU. Small bright noise that is alone is thrown away.

**Why marching cubes?** It is the classic method that turns a 3D array into a surface made of triangles. It was invented for CT and MR data [5]. We give it the real voxel size, so the 3D model keeps the right shape (the slices are 5 mm apart, but the pixels are only 0.83 mm).

**What happens when we change the values?**
- *Window width*: a wider window (2000 HU) shows all tissues, but with low contrast (everything looks grey). A narrower window (400 HU) gives strong contrast, but everything outside the window becomes pure black or pure white.
- *Window level*: a higher level makes the image darker, so only dense things stay bright. A lower level makes the image brighter.
- *Bone threshold (low value)*: going from 250 to 100 HU adds cartilage, calcified vessels and voxels at the edge of the body (more volume, many extra small pieces). Going from 250 to 400 HU loses the thin ribs and the soft bone inside the vertebrae. 250 HU was the best middle choice. The cell below prints the voxel counts for several thresholds (run it in Colab with the data).
- *MIN_VOL*: a smaller value keeps more noise. A larger value can delete small real bones.
- *MAX_HOLE*: without hole filling, the ribs are hollow tubes (the marrow inside is darker than the hard bone). If this value is too big, the lungs would be filled too.

**Challenges**
- The distance between slices (5 mm) is 6 times bigger than the pixel size. So the 3D model looks "stepped" from head to foot. This comes from the data, not from our method.
- The sternum and the rib cartilage have HU values close to soft tissue, so they are partly missing. A lower threshold would add them, but it would also add noise.

#### Extra 1.1 – what happens when the bone threshold changes?
Run this cell in Colab (it needs the DICOM volume). It prints how many bone voxels we get for different thresholds and shows three of them on one slice.

In [ ]:
# ---------- 1.1 (extra): what happens when the bone threshold changes? ----------
# The professor suggests "thresholding" for the ribcage phantom. Here we vary the two
# hysteresis thresholds and count how much "bone" we get, to justify the values 250 / 400 HU.
from skimage.filters import apply_hysteresis_threshold

print(f"{'low (HU)':>9} {'high (HU)':>10} | {'voxels':>9} {'objects':>8} {'volume cm3':>11}")
print("-" * 55)
for low, high in [(100, 400), (150, 400), (250, 400), (350, 400), (250, 300), (250, 600), (400, 400)]:
    m = apply_hysteresis_threshold(vol_body, low=low, high=high)
    m = remove_small_objects(m, min_size=int(MIN_VOL / voxel_mm3))
    _, n_obj = label(m)
    print(f"{low:>9} {high:>10} | {m.sum():>9,} {n_obj:>8} {m.sum() * voxel_mm3 / 1000:>11.0f}")

# Visual check on one slice for three "low" thresholds (high fixed at 400 HU)
z = vol_hu.shape[0] // 2
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, low in zip(axes, [100, 250, 400]):
    m = apply_hysteresis_threshold(vol_body, low=low, high=400)
    ax.imshow(vol_hu[z], cmap="gray", vmin=400 - 900, vmax=400 + 900)   # bone window
    ax.contour(m[z], levels=[0.5], colors="r", linewidths=0.8)
    ax.set_title(f"Slice {z}: bone if HU >= {low} (connected to >= 400)")
    ax.set_xlabel("X (pixels)"); ax.set_ylabel("Y (pixels)")
plt.tight_layout(); plt.show()


### 1.2 Create a simple and homogeneous phantom (1.5 points)

Now create from scratch a more simple 2D phantom which contains a fake (bright) lesion as shown below:

<img src='https://github.com/sacumesh/medical-imagining-labs/blob/sachiththa/P1/images_4_notebook/PhantomImage.png?raw=1' width="400">

- Phantom size: 256 x 256 pixels.
- Phantom must contain 3 tissues:
    - Background = 0 pixel intensity value
    - Tissue 1 = 1 (radius = 100 pixels)
    - Tissue 2 (lesion) = 2 (radius = 5 pixels)
    
Help: To create the circunferences, you can use the function ``create_circular_mask``, or define your own function. The lesion can be placed anywhere within the phantom, except at the exact center.

Once constructed, add a second lesion in the same row (same size and pixel intensity), plot the original phantom and a pixel intensity (horizontal) profile along the synthetic lesions, similarly as shown above (but with two lesions). Always remember to include the units of the axis when ploting a graph.

``Questions``: What challenges did you face? How did you deal with them?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon
from skimage.transform import iradon

In [ ]:
def create_circular_mask(h, w, center=None, radius=None):
    # If no center is provided, default to the middle of the image
    if center is None:
        center = (w // 2, h // 2)
    # If no radius is provided, make it touch the closest border
    if radius is None:
        radius = min(center[0], center[1], w - center[0], h - center[1])

    # Create grid coordinates for Y (rows) and X (columns)
    Y, X = np.ogrid[:h, :w]
    # Calculate distance from each coordinate to the center
    dist_squared = (X - center[0]) ** 2 + (Y - center[1]) ** 2
    # Return True where points are inside the circle's radius
    return dist_squared <= radius ** 2

def fill_circular_segment(canvas, center=None, radius=None, val=0):
    h, w = canvas.shape[:2]
    if center is None:
        center = (w // 2, h // 2)
    if radius is None:
        radius = min(center[0], center[1], w - center[0], h - center[1])

    # Generate the circular binary mask
    mask = create_circular_mask(h, w, center, radius)

    # Fill the circle area on the canvas with the specified value
    return np.where(mask, val, canvas)

In [ ]:
import matplotlib.patches as mpatches

# Define the center coordinates of our phantom and the row for the lesions
phantom_size = 256
phantom_center_x, phantom_center_y = phantom_size // 2, phantom_size // 2
lesion_y = 64

# Put the lesions 30 pixels to the left and right of the centre
lesion_offset = 30
lesion_left_x = phantom_center_x - lesion_offset
lesion_right_x = phantom_center_x + lesion_offset

# Big circle = tissue 1 (value 1, radius 100) on a 256 x 256 grid
phantom = create_circular_mask(phantom_size, phantom_size, center=(phantom_center_x, phantom_center_y), radius=100).astype(float)

# Define the size, position, and intensity value (2) for the two lesions
lesions = [
    {"center": (lesion_left_x, lesion_y), "radius": 5, "val": 2},
    {"center": (lesion_right_x, lesion_y), "radius": 5, "val": 2}
]

# Draw each lesion onto our phantom canvas
for lesion in lesions:
    phantom = fill_circular_segment(phantom, **lesion)

# Take the row of pixels that goes through both lesions
profile = phantom[lesion_y, :]

# Set up a side-by-side plot layout for the image and the profile graph
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 5))

# Plot 1: Show the synthetic phantom with the horizontal line of interest
im = ax1.imshow(phantom, cmap='gray', vmin=0, vmax=2)
ax1.set_title("Synthetic Phantom")
line_handle = ax1.axhline(y=lesion_y, color='red', linestyle='--', linewidth=1.5, label=f'Profile Line (y={lesion_y})')
ax1.set_xlabel("X (pixels)")
ax1.set_ylabel("Y (pixels)")

# Create a colored legend to explain what the pixel intensity values represent
bg_patch = mpatches.Patch(color='black', label='0: Background')
tissue1_patch = mpatches.Patch(color='gray', label='1: Tissue 1')
lesion_patch = mpatches.Patch(color='white', label='2: Lesion')

# Place the legend neatly to the right of the phantom image
ax1.legend(
    handles=[line_handle, bg_patch, tissue1_patch, lesion_patch],
    loc='upper left',
    bbox_to_anchor=(1.04, 1),
    framealpha=0.8
)
ax1.axis('on')

# Plot 2: Graph the pixel values across the horizontal line to see intensity changes
ax2.plot(profile, color='red', linewidth=2)
ax2.set_title(f"Intensity Profile at y = {lesion_y}")
ax2.set_xlabel("X (pixels)")
ax2.set_ylabel("Pixel Intensity Value")
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.set_ylim(-0.2, 2.2)

plt.tight_layout()
plt.show()

#### Discussion 1.2 – What, how and why

**What we did.** We made a 256 × 256 image with 3 values: background = 0, one big circle (tissue 1 = 1, radius 100 pixels) and two small bright lesions (value 2, radius 5 pixels). The two lesions are on the same row `y = 64`, at x = 98 and x = 158. We plotted the phantom and the intensity profile along this row, with units on the axes. The profile shows the 3 levels: 0 outside, a flat line at 1 inside the tissue, and two narrow peaks at 2 where the lesions are.

**How.** A pixel is inside a circle if `(x − cx)² + (y − cy)² ≤ r²`. `np.ogrid` gives the x and y position of every pixel, so we can build the mask in one line. `np.where(mask, value, canvas)` paints the value onto the image. We paint the lesions after the tissue, so the lesions are on top.

**Why.** A phantom is a test object where we know the true values exactly. This is the only way to measure how good a reconstruction is. In 1.4 we compare the reconstructed image with this true image, pixel by pixel. The Shepp–Logan phantom in 1.5 is the famous version of the same idea [6].

**Question – What challenges did we face? How did we deal with them?**
1. *Order of x and y.* NumPy arrays are written as `[row, column]`, which is `[y, x]`. But a centre is usually written as `(x, y)`. If we mix them up, the lesion goes to the wrong place. So we always give `center = (x, y)` and use `X - center[0]` and `Y - center[1]` inside the function.
2. *Not in the exact centre.* A lesion in the centre of rotation gives a flat line in the sinogram, not a sine wave. So we put the lesions away from the centre, in the upper part of the image. Then their sinograms are clearly curved.
3. *Pixels are not round.* A circle with radius 5 pixels is only about 10 pixels wide, so it looks like a small octagon. This is normal. It is the same partial-volume effect that we see in real CT.


### 1.3 Create the projections of the phantom (2 points)

Generate the projections of the previously generated 2D phantom using the ``radon`` transform from ``skimage.transform``. Play with the different parameters to optimise sinogram. Then, show the sinogram of the projections.

In this section, considered the following approch:

Use different number of projections (i.e. 4, 20, 50, 100, 200, 300, 360) considering that in all cases the simulated projections cover the 360º. For example, 100 projections are taken in the range from 0º to 360º.


``Questions``: How do the sinograms changes with number of projections? What is the effect of increasing/decrasing the number of projections?

In [ ]:
from skimage.transform import radon

# Configuration parameters
n_projections = 100
theta = np.linspace(0.0, 360.0, n_projections, endpoint=False)

# Compute the forward Radon transform (sinogram)
sinogram = radon(phantom, theta=theta)

# Visualize the original phantom alongside its sinogram
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))

# Original Phantom Plot
ax1.set_title("Original Phantom")
ax1.imshow(phantom, cmap='gray')
ax1.axis('on')

# Sinogram Plot
ax2.set_title("Radon Transform (Sinogram)")
ax2.set_xlabel("Projection Angle (degrees)")
ax2.set_ylabel("Sensor Position (pixels)")

# Display sinogram with projection angles properly mapped on the x-axis
im = ax2.imshow(
    sinogram,
    cmap='gray',
    extent=(0.0, 360.0, 0, sinogram.shape[0]),
    aspect='auto'
)

plt.tight_layout()
plt.show()

#### Extra 1.3 A – different numbers of projections (4 … 360)

In [ ]:
# ---- 1.3 (extra A): sinograms and reconstructions for different numbers of projections ----
from skimage.transform import radon, iradon
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse

N_PROJ_LIST = [4, 20, 50, 100, 200, 300, 360]   # all cover the full 0-360 degree range
d_range = phantom.max() - phantom.min()
sweep = {}

fig, axes = plt.subplots(3, len(N_PROJ_LIST), figsize=(4 * len(N_PROJ_LIST), 12))
print(f"{'n_proj':>7} | {'angle step (deg)':>16} | {'sinogram shape':>15} | {'MSE':>7} | {'PSNR (dB)':>9} | {'SSIM':>6}")
print("-" * 80)

for col, n in enumerate(N_PROJ_LIST):
    theta_n = np.linspace(0.0, 360.0, n, endpoint=False)
    sino_n = radon(phantom, theta=theta_n)                       # forward projection -> sinogram
    rec_n = iradon(sino_n, theta=theta_n, filter_name="ramp",    # simple FBP so we can SEE the effect
                   output_size=phantom.shape[0])
    sweep[n] = (mse(phantom, rec_n), psnr(phantom, rec_n, data_range=d_range), ssim(phantom, rec_n, data_range=d_range))
    print(f"{n:>7} | {360.0 / n:>16.2f} | {str(sino_n.shape):>15} | {sweep[n][0]:>7.4f} | {sweep[n][1]:>9.2f} | {sweep[n][2]:>6.3f}")

    # Row 1: sinogram (x = projection angle, y = detector position)
    ax = axes[0, col]
    ax.imshow(sino_n, cmap="gray", aspect="auto", extent=(0.0, 360.0, 0, sino_n.shape[0]))
    ax.set_title(f"Sinogram, {n} projections")
    ax.set_xlabel("Projection angle (degrees)")
    ax.set_ylabel("Detector position (pixels)")

    # Row 2: FBP reconstruction from that sinogram
    ax = axes[1, col]
    ax.imshow(rec_n, cmap="gray", vmin=0, vmax=2)
    ax.axhline(y=lesion_y, color="red", linestyle="--", linewidth=1)
    ax.set_title(f"FBP (ramp), {n} projections")
    ax.set_xlabel("X (pixels)")
    ax.set_ylabel("Y (pixels)")

    # Row 3: profile through the lesions
    ax = axes[2, col]
    ax.plot(phantom[lesion_y, :], color="black", linewidth=2, label="Original")
    ax.plot(rec_n[lesion_y, :], color="tab:red", alpha=0.85, label=f"{n} proj.")
    ax.set_ylim(-0.5, 3.0)
    ax.set_xlabel("X (pixels)")
    ax.set_ylabel("Pixel intensity (a.u.)")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper right", fontsize=8)

plt.suptitle("Effect of the number of projections (all covering 0-360 degrees)", fontsize=16, y=1.0)
plt.tight_layout()
plt.show()

# Summary curve: quality vs number of projections
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(N_PROJ_LIST, [sweep[n][1] for n in N_PROJ_LIST], "o-")
ax[0].set_xlabel("Number of projections"); ax[0].set_ylabel("PSNR (dB)"); ax[0].grid(ls="--", alpha=.5)
ax[1].plot(N_PROJ_LIST, [sweep[n][2] for n in N_PROJ_LIST], "o-", color="tab:green")
ax[1].set_xlabel("Number of projections"); ax[1].set_ylabel("SSIM"); ax[1].grid(ls="--", alpha=.5)
plt.suptitle("Reconstruction quality vs number of projections (ramp FBP)")
plt.tight_layout(); plt.show()


#### Extra 1.3 B – playing with the other parameters of `radon` (angular range, `circle`)

In [ ]:
# ---- 1.3 (extra B): "play with the parameters" of radon() ----
# (1) angular range 0-180 vs 0-360: for a parallel beam the projection at angle a+180 is the
#     mirror image of the one at angle a, so 0-360 contains every projection twice.
# (2) circle=True (default) vs circle=False: with circle=False the image is zero-padded so that
#     the corners of the square image are also projected (longer detector).
n = 100
theta_180 = np.linspace(0.0, 180.0, n, endpoint=False)
theta_360 = np.linspace(0.0, 360.0, n, endpoint=False)
sino_180 = radon(phantom, theta=theta_180)
sino_360 = radon(phantom, theta=theta_360)
sino_nocircle = radon(phantom, theta=theta_180, circle=False)

print("sinogram shape (detector pixels, angles):")
print("   0-180, circle=True :", sino_180.shape)
print("   0-360, circle=True :", sino_360.shape)
print("   0-180, circle=False:", sino_nocircle.shape, " <- detector grows to the image diagonal")
# check the mirror property: projection at 0 deg == flipped projection at 180 deg
p0 = radon(phantom, theta=[0.0])[:, 0]
p180 = radon(phantom, theta=[180.0])[:, 0]
print("max |p(0) - flip(p(180))| =", np.abs(p0 - p180[::-1]).max().round(6), " (0 => identical information)")

rec_180 = iradon(sino_180, theta=theta_180, filter_name="ramp", output_size=phantom.shape[0])
rec_360 = iradon(sino_360, theta=theta_360, filter_name="ramp", output_size=phantom.shape[0])
print(f"PSNR 100 proj over 0-180: {psnr(phantom, rec_180, data_range=d_range):.2f} dB")
print(f"PSNR 100 proj over 0-360: {psnr(phantom, rec_360, data_range=d_range):.2f} dB")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, s, ttl, rng in [(axes[0], sino_180, "0-180 deg, circle=True", 180), (axes[1], sino_360, "0-360 deg, circle=True", 360),
                        (axes[2], sino_nocircle, "0-180 deg, circle=False", 180)]:
    ax.imshow(s, cmap="gray", aspect="auto", extent=(0, rng, 0, s.shape[0]))
    ax.set_title(f"Sinogram: {ttl}\nshape {s.shape}")
    ax.set_xlabel("Projection angle (degrees)"); ax.set_ylabel("Detector position (pixels)")
plt.tight_layout(); plt.show()


#### Discussion 1.3 – What, how and why

**What a sinogram is.** The Radon transform (`radon`) adds up all the pixel values along straight lines through the image, for one angle [7]. Each angle gives one column of the sinogram (x-axis = projection angle, y-axis = position on the detector). A bright point that is not in the centre of the image draws a **sine wave** in the sinogram. This is why it is called a "sinogram". In our sinogram we see one wide bright band (the big circle; it is in the centre, so it does not move) and two thin sine waves (the two lesions). The two lesions have the same size and the same distance from the centre, so their two curves have the same brightness and the same height. The curves cross each other at 0° and 180°, because the two lesions are on the same row.

**How.** `theta = np.linspace(0, 360, n, endpoint=False)` gives `n` angles with equal steps around the full circle. `endpoint=False` is important: 360° is the same direction as 0°. With `endpoint=True`, that direction would be measured two times.

**Question – How do the sinograms change with the number of projections? What happens when we use more or fewer projections?**
(See the table and the figures in the next two cells: we used 4 to 360 projections, and reconstructed each one with the ramp filter.)

| projections | angle step | MSE | PSNR (dB) | SSIM |
|---|---|---|---|---|
| 4 | 90° | 0.909 | 6.4 | 0.17 |
| 20 | 18° | 0.063 | 18.0 | 0.30 |
| 50 | 7.2° | 0.0110 | 25.6 | 0.47 |
| 100 | 3.6° | 0.0034 | 30.7 | 0.67 |
| 200 | 1.8° | 0.0014 | 34.6 | 0.88 |
| 300 | 1.2° | 0.0011 | 35.5 | 0.94 |
| 360 | 1.0° | 0.0010 | 35.8 | 0.96 |

- **How the sinogram looks.** With 4 projections, the sinogram is only 4 wide columns. There is no sine wave. With 20 projections, we see a "dotted" curve. From about 100 projections, the curves are smooth. Between 200 and 360 projections, the sinogram looks almost the same.
- **Reconstruction.** With few projections we get strong **streak artifacts** (lines across the image). With 4 projections the image is a star of lines and we cannot see the lesions. With 20 there is still a web of lines. With 50–100 a faint pattern is still there in the flat areas. When the angle step is too big (under-sampling), sharp edges and small objects are not shown correctly. The fix is to take as many projections per rotation as possible [8]. With FBP, too few angles always give streaks [9].
- **Less and less gain.** At first, the error goes down very fast (MSE is 14 times smaller from 4 → 20). Later it goes down slowly (from 300 → 360, SSIM only moves from 0.94 to 0.96). Sampling theory says that an image with N detector pixels needs about π/2 × N projections over 180° [18, 19]. For 256 pixels this is about 400. So even 360 projections over 360° is a little too few, and the background is not perfectly flat.
- **Fewer projections:** less data, less time, and in a real scanner **less radiation dose**. But more streaks and more noise. A real CT scanner uses a thousand or more projections. Using 10 to 100 times fewer gives streaks, unless special algorithms are used [10].
- **More projections:** a better image. But every projection costs time and dose. Above the sampling limit, there is almost no more gain.

**Playing with the other parameters of `radon` (cell "extra B")**
- *0–180° vs 0–360°.* For a parallel beam, the projection at angle a + 180° is the mirror image of the projection at angle a. (We checked this: after flipping, they are the same.) So a 0–360° scan contains every projection two times. With 100 projections, 0–180° gives **PSNR 34.6 dB**, but 0–360° gives only **30.7 dB**. This is because the 0–360° scan really has only 50 different directions. 200 projections over 360° give the same result as 100 over 180°. We still use 0–360° because the task asks for it. Also, a real scanner turns a full circle, and a fan beam needs more than 180°.
- *`circle=True` vs `circle=False`.* By default, `radon` only projects the circle that fits inside the image (detector = 256 pixels). With `circle=False`, the image is padded so the corners are also included (detector = 363 pixels). For our round phantom, the extra pixels are all zero.
- *Image size.* Sinogram values are sums of pixels along a line, so they grow with the image size (maximum ≈ 210 for a 256-pixel image, ≈ 420 for 512 pixels).

**Conclusion.** When the number of projections goes up, the sinogram becomes smoother and the streaks go away. About 200 projections already give an almost perfect image (SSIM 0.88). 100 projections is a good middle choice, and this is what we use in 1.4.

### 1.4 Reconstruction with Filtered Back Projection (FBP) (2.5 points)

FBP is one of the most simple reconstruction methods to reconstruct images in CT. Explore the ``iradon`` from ``skimage.transform`` using the different filters available (Ramp filter (default), Shepp-logan, Cosine, Hamming, Hann).

Make use of the ``matplotlib`` to show the original and reconstructed images of the phantom and compare the pixel intensity signal across the lesion profile (similarly to section 2.1).In addition, provide evaluation metric you could consider useful for this purpose (image difference, mean square error (MSE), peak signal to noise ratio (PSNR), structural index similarity (SSIM), etc.). Then, discuss the results.

In [ ]:
from skimage.transform import iradon
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Set up the angles for the CT scan projections (0 to 360 degrees)
n_projections = 100
theta = np.linspace(0.0, 360.0, n_projections, endpoint=False)

# Step 2: Create the forward sinogram using the Radon transform
sinogram = radon(phantom, theta=theta)

# Get the size of the original phantom to match the reconstruction
N_size = phantom.shape[0]

# Calculate intensity range for proper metric calculation
d_range = phantom.max() - phantom.min()

# Step 3: Define reconstruction filters to compare
FILTERS = ["ramp", "shepp-logan", "cosine", "hamming", "hann"]
recons = {}
metrics = {}

# Run Filtered Back-Projection (FBP) for each filter type
for f in FILTERS:
    rec = iradon(sinogram, theta=theta, filter_name=f, output_size=N_size)
    recons[f] = rec
    # Compute similarity and error metrics
    metrics[f] = {
        "mse": mse(phantom, rec),
        "psnr": psnr(phantom, rec, data_range=d_range),
        "ssim": ssim(phantom, rec, data_range=d_range),
        "mae": np.mean(np.abs(phantom - rec))
    }

# Step 4: Compute Unfiltered Back-Projection (baseline without any filter)
bp_raw = iradon(sinogram, theta=theta, filter_name=None, output_size=N_size)
print("raw BP range:", bp_raw.min().round(2), bp_raw.max().round(2), "  <- not pixel values, a sum of line integrals")
# BP has its own scale, so we fit it to the phantom (best straight line a*BP + b) before comparing
a, b = np.polyfit(bp_raw.ravel(), phantom.ravel(), 1)
bp = a * bp_raw + b

# Compute metrics for the unfiltered back-projection
metrics["Unfiltered BP"] = {
    "mse": mse(phantom, bp),
    "psnr": psnr(phantom, bp, data_range=d_range),
    "ssim": ssim(phantom, bp, data_range=d_range),
    "mae": np.mean(np.abs(phantom - bp))
}

# Put all reconstructions together
all_recons = {"Unfiltered BP": bp}
all_recons.update(recons)

# Step 5: Plot results side-by-side (Original, Reconstruction, Profile, Metrics)
n_rows = len(all_recons)
fig, axes = plt.subplots(n_rows, 4, figsize=(24, 5 * n_rows))

for idx, (name, rec_img) in enumerate(all_recons.items()):
    # Column 1: Display original phantom with profile reference line
    ax_orig = axes[idx, 0]
    ax_orig.imshow(phantom, cmap="gray")
    ax_orig.axhline(y=lesion_y, color="red", linestyle="--", linewidth=1.5)
    ax_orig.set_title("Original Phantom", fontsize=14)
    ax_orig.set_xlabel("X (pixels)", fontsize=12)
    ax_orig.set_ylabel("Y (pixels)", fontsize=12)

    # Column 2: Display reconstructed image with profile line
    ax_rec = axes[idx, 1]
    ax_rec.imshow(rec_img, cmap="gray")
    ax_rec.axhline(y=lesion_y, color="red", linestyle="--", linewidth=1.5)
    ax_rec.set_title(f"{name} Reconstruction", fontsize=14)
    ax_rec.set_xlabel("X (pixels)", fontsize=12)
    ax_rec.set_ylabel("Y (pixels)", fontsize=12)

    # Column 3: Plot the intensity profile along the lesion row
    ax_prof = axes[idx, 2]
    ax_prof.plot(phantom[lesion_y, :], label="Original Phantom", color="black", linewidth=2.0)
    ax_prof.plot(rec_img[lesion_y, :], label=f"{name}", color="tab:red", linestyle="-", alpha=0.85)
    ax_prof.set_title(f"Intensity Profile (y = {lesion_y})", fontsize=14)
    ax_prof.set_xlabel("X (pixels)", fontsize=12)
    ax_prof.set_ylabel("Intensity", fontsize=12)
    ax_prof.set_ylim(-0.5, 2.5)
    ax_prof.grid(True, linestyle="--", alpha=0.5)
    ax_prof.legend(loc="upper right", fontsize=10)

    # Column 4: Render evaluation metrics as text
    ax_met = axes[idx, 3]
    ax_met.axis("off")
    if name in metrics:
        m = metrics[name]
        text_str = (
            f"\nMetrics:\n\n"
            f"• MSE: {m["mse"]:.4f}\n"
            f"• PSNR: {m["psnr"]:.2f} dB\n"
            f"• SSIM: {m["ssim"]:.3f}\n"
            f"• MAE: {m["mae"]:.4f}"
        )
        ax_met.text(0.1, 0.5, text_str, fontsize=14, va='center', ha='left', family='monospace')

# Position the suptitle and add top spacing to avoid overlap with the subplots
plt.suptitle("Row-by-Row Comparison: Original, Reconstruction, Profiles & Metrics", fontsize=18, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

#### Extra 1.4 – why the filters differ, difference images, and the `interpolation` parameter

In [ ]:
# ---- 1.4 (extra): why the filters behave differently + image-difference maps ----
from skimage.transform.radon_transform import _get_fourier_filter

# (1) Frequency response of each filter (what iradon multiplies the projections with)
fig, ax = plt.subplots(figsize=(7, 4))
for f in FILTERS:
    resp = _get_fourier_filter(512, f)[:, 0]
    ax.plot(np.linspace(0, 1, 256), resp[:256], label=f)
ax.set_xlabel("Spatial frequency (fraction of Nyquist)"); ax.set_ylabel("Filter gain")
ax.set_title("FBP filters: ramp vs windowed ramps"); ax.legend(); ax.grid(ls="--", alpha=.5)
plt.tight_layout(); plt.show()

# (2) Image difference |original - reconstruction| for every filter (same colour scale)
fig, axes = plt.subplots(1, len(FILTERS), figsize=(4 * len(FILTERS), 4.3))
for ax, f in zip(axes, FILTERS):
    diff = np.abs(phantom - recons[f])
    im = ax.imshow(diff, cmap="magma", vmin=0, vmax=0.3)
    ax.set_title(f"{f}: |diff|, mean = {diff.mean():.4f}")
    ax.set_xlabel("X (pixels)"); ax.set_ylabel("Y (pixels)")
fig.colorbar(im, ax=axes, fraction=0.015, label="absolute error (a.u.)")
plt.show()

# (3) Interpolation parameter of iradon: nearest vs linear vs cubic (ramp filter)
for interp in ["nearest", "linear", "cubic"]:
    r = iradon(sinogram, theta=theta, filter_name="ramp", interpolation=interp, output_size=N_size)
    print(f"interpolation={interp:8s}: MSE={mse(phantom, r):.4f}  PSNR={psnr(phantom, r, data_range=d_range):.2f} dB  SSIM={ssim(phantom, r, data_range=d_range):.3f}")


#### Discussion 1.4 – What, how and why

**What FBP is.** Back projection takes one projection and "paints" it back over the whole image, in the direction it was measured. We do this for every angle and add all of them up. If we stop here, the image is very blurry (see the row "Unfiltered BP": we can hardly see the lesions, and the profile is a round hump, not a flat line). The image is blurry because each point gets some signal from every direction. FBP adds one extra step to fix this. Before painting back, each projection goes through a **ramp filter**. The ramp filter makes the fine details (high frequencies) stronger. This removes the blur [7]. FBP is one of the fastest ways to make a CT image. The filter is the only setting we can change [11].

**The five filters.** The ramp filter (also called Ram-Lak) gets stronger in a straight line as the frequency goes up. So it also makes the noise stronger. The other four filters are the same ramp, but multiplied by a "window". The window makes the highest frequencies weaker again. Shepp-Logan uses a sinc shape. Cosine uses a cosine shape. Hamming and Hann use their own window shapes [12]. From the sharpest to the softest: ramp < Shepp-Logan < cosine < Hamming ≈ Hann (see the plot of the filters in the "extra" cell).

**Results (100 projections, no noise):**

| filter | MSE | PSNR (dB) | SSIM | MAE |
|---|---|---|---|---|
| Unfiltered BP (rescaled) | 0.0866 | 16.65 | 0.440 | 0.2543 |
| ramp | 0.0034 | 30.71 | 0.669 | 0.0364 |
| shepp-logan | 0.0029 | 31.38 | 0.718 | 0.0319 |
| cosine | 0.0026 | 31.91 | 0.802 | 0.0253 |
| hamming | 0.0026 | 31.90 | 0.844 | 0.0224 |
| hann | 0.0026 | 31.80 | 0.855 | 0.0216 |

**Discussion**
- *Filtered vs unfiltered.* The raw unfiltered BP has values from 0 to 315. This is because it is a sum of many pixels along a line, not a pixel value. To compare it with the phantom, we first fit it to the phantom range (`a·BP + b`). Even after this, its MSE (0.087) is 25 times worse than the ramp (0.0034). Rescaling cannot remove the blur.
- *Ramp = sharp, but with small waves.* The ramp profile follows the true profile well: the flat part is at 1 and the lesion peaks reach 2 (even a little above). But there are small waves next to every edge (about ±0.1–0.2 outside the phantom, and a faint pattern inside). These waves are called ringing. They come from the sharp cut-off of the ramp filter, and from using only 100 projections.
- *Windowed filters are smoother.* Shepp-Logan → cosine → Hamming → Hann make the high frequencies weaker step by step. So the waves get smaller and SSIM goes up (0.67 → 0.86). Hann has the lowest MAE and the highest SSIM. Cosine has the best PSNR. In the difference images, the ramp error is a fine texture everywhere, but the Hann error is only on the edges. The price is a slightly softer edge: with Hann, the jump from 1 to 2 takes about one pixel more, and the peak is about 1.98 instead of 2. For our 10-pixel lesions this is not a problem. A very small lesion (1–2 pixels) could be hidden by a soft filter.
- *What the metrics mean.* MSE and MAE = the average error per pixel (lower is better). PSNR = the same error written in decibels (higher is better). SSIM looks at local shape, contrast and brightness, so it is closer to how people see images. Two images can have the same PSNR but very different SSIM [13]. SSIM was made because MSE and PSNR are bad at seeing errors in shape and structure [14]. In our table, SSIM shows the difference between the filters best (0.67 → 0.86), while PSNR changes only by about 1 dB. This is because SSIM notices the small waves in the flat areas, just like our eyes do.


### 1.5 Sheep logan phantom (2 points)

So far we have used a quite simple phantom with only 2 tissues. Now, repeat the prior analysis for the **noisy sinogram** for one case (approach, filter,...) using a (slightly) more advanced head test object, the Sheep-logan phantom. Do you observe particular differences between the reconstrucction results from both phantoms?


<img src='https://github.com/sacumesh/medical-imagining-labs/blob/sachiththa/P1/images_4_notebook/SheepLoganPhantom.png?raw=1' width="1000">



In order to create **noisy sinograms**, you can add Poison noise (``np.random.poisson``; perhaps with lam = 10) to your noise-free sinograms (not to the image!) and reconstruct the phantom images with the different filters. Plot the reconstruction image and the (horizontal) intensity profile along specific row. Then, comment on the effects of the filters on the reconstructed phatom images.


In [ ]:
from skimage.data import shepp_logan_phantom
from skimage.transform import iradon, radon
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Load the standard Shepp-Logan head phantom
logan_phantom = shepp_logan_phantom()
phantom = logan_phantom

# Select the row across the center of the phantom to extract intensity profiles
lesion_y = 175

# Step 2: Set up 200 projection angles covering a full 360-degree rotation
n_projections = 200
theta = np.linspace(0.0, 360.0, n_projections, endpoint=False)   # 360 deg = 0 deg, do not repeat it

# Step 3: Compute the clean forward radon transform (sinogram)
sinogram = radon(phantom, theta=theta)

# Step 4: Add Poisson noise (using lambda = 10) to simulate a real noisy CT scan
np.random.seed(0)   # same random noise every time we run the cell
noisy_sinogram = sinogram + np.random.poisson(lam=10, size=sinogram.shape)

# Get the dimensions of the original phantom for reconstruction size matching
N_size = phantom.shape[0]

# Determine the intensity range of the phantom to calculate quality metrics
d_range = phantom.max() - phantom.min()

# Step 5: Define the reconstruction filters we want to test and compare
FILTERS = ["ramp", "shepp-logan", "cosine", "hamming", "hann"]
recons = {}
metrics = {}

# Run Filtered Back-Projection (FBP) for each filter using the noisy sinogram
for f in FILTERS:
    rec = iradon(noisy_sinogram, theta=theta, filter_name=f, output_size=N_size)
    recons[f] = rec
    # Calculate error and similarity metrics compared to the original clean phantom
    metrics[f] = {
        "mse": mse(phantom, rec),
        "psnr": psnr(phantom, rec, data_range=d_range),
        "ssim": ssim(phantom, rec, data_range=d_range),
        "mae": np.mean(np.abs(phantom - rec))
    }

# Step 6: Compute the baseline Unfiltered Back-Projection from the noisy sinogram
bp_raw = iradon(noisy_sinogram, theta=theta, filter_name=None, output_size=N_size)
print("raw BP range:", bp_raw.min().round(2), bp_raw.max().round(2), "  <- not pixel values, a sum of line integrals")
# BP has its own scale, so we fit it to the phantom (best straight line a*BP + b) before comparing
a, b = np.polyfit(bp_raw.ravel(), phantom.ravel(), 1)
bp = a * bp_raw + b

# Calculate performance metrics for the unfiltered back-projection
metrics["Unfiltered BP"] = {
    "mse": mse(phantom, bp),
    "psnr": psnr(phantom, bp, data_range=d_range),
    "ssim": ssim(phantom, bp, data_range=d_range),
    "mae": np.mean(np.abs(phantom - bp))
}

# Combine all our reconstructed results into a single dictionary
all_recons = {"Unfiltered BP": bp}
all_recons.update(recons)

# Step 7: Plot a detailed grid comparing all reconstruction methods
n_rows = len(all_recons)
fig, axes = plt.subplots(n_rows, 5, figsize=(36, 6 * n_rows))

for idx, (name, rec_img) in enumerate(all_recons.items()):
    # Column 1: Show clean original phantom with reference line
    ax_orig = axes[idx, 0]
    ax_orig.imshow(phantom, cmap="gray")
    ax_orig.axhline(y=lesion_y, color="red", linestyle="--", linewidth=1.5)
    ax_orig.set_title("Original Phantom", fontsize=16)
    ax_orig.set_xlabel("X (pixels)", fontsize=14)
    ax_orig.set_ylabel("Y (pixels)", fontsize=14)

    # Column 2: Show the noisy sinogram used for reconstruction
    ax_sino = axes[idx, 1]
    ax_sino.imshow(noisy_sinogram, cmap="gray", aspect='auto', extent=(0.0, 360.0, 0, noisy_sinogram.shape[0]))
    ax_sino.set_title("Noisy Sinogram", fontsize=16)
    ax_sino.set_xlabel("Projection Angle (degrees)", fontsize=14)
    ax_sino.set_ylabel("Sensor Position", fontsize=14)

    # Column 3: Show reconstructed slice using the current filter
    ax_rec = axes[idx, 2]
    ax_rec.imshow(rec_img, cmap="gray")
    ax_rec.axhline(y=lesion_y, color="red", linestyle="--", linewidth=1.5)
    ax_rec.set_title(f"{name} Reconstruction", fontsize=16)
    ax_rec.set_xlabel("X (pixels)", fontsize=14)
    ax_rec.set_ylabel("Y (pixels)", fontsize=14)

    # Column 4: Compare original vs reconstructed intensity profiles with automatic individual scaling
    ax_prof = axes[idx, 3]
    original_profile = phantom[lesion_y, :]
    reconstructed_profile = rec_img[lesion_y, :]

    ax_prof.plot(original_profile, label="Original Phantom", color="black", linewidth=2.0)
    ax_prof.plot(reconstructed_profile, label=f"{name}", color="tab:red", linestyle="-", alpha=0.85)
    ax_prof.set_title(f"Intensity Profile (y = {lesion_y})", fontsize=16)
    ax_prof.set_xlabel("X (pixels)", fontsize=14)
    ax_prof.set_ylabel("Intensity", fontsize=14)

    # Let matplotlib automatically scale the limits individually for this subplot
    ax_prof.autoscale(enable=True, axis='y', tight=False)

    ax_prof.grid(True, linestyle="--", alpha=0.5)
    ax_prof.legend(loc="upper right", fontsize=12)

    # Column 5: Print numerical metrics next to each image row
    ax_met = axes[idx, 4]
    ax_met.axis("off")
    if name in metrics:
        m = metrics[name]
        text_str = (
            f"\nMetrics:\n\n"
            f"• MSE: {m['mse']:.4f}\n"
            f"• PSNR: {m['psnr']:.2f} dB\n"
            f"• SSIM: {m['ssim']:.3f}\n"
            f"• MAE: {m['mae']:.4f}"
        )
        ax_met.text(0.1, 0.5, text_str, fontsize=16, va='center', ha='left', family='monospace')

plt.suptitle("Shepp-Logan Phantom: Noisy Sinogram Reconstructions", fontsize=22, y=1.0)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

#### Extra 1.5 – what happens when the noise level λ changes?

In [ ]:
# ---- 1.5 (extra): what happens when the noise level (lambda) changes? ----
# Poisson(lam) has mean = lam and standard deviation = sqrt(lam), so a larger lam means
# MORE noise (and a larger constant offset, which the ramp filter removes anyway).
LAMS = [0, 1, 10, 50, 200]
clean_sino = radon(phantom, theta=theta)
rows = {"ramp": [], "hann": []}

fig, axes = plt.subplots(2, len(LAMS), figsize=(4.2 * len(LAMS), 8.5))
for col, lam in enumerate(LAMS):
    np.random.seed(0)
    ns = clean_sino + (np.random.poisson(lam=lam, size=clean_sino.shape) if lam > 0 else 0)
    for row, f in enumerate(["ramp", "hann"]):
        r = iradon(ns, theta=theta, filter_name=f, output_size=N_size)
        rows[f].append((mse(phantom, r), psnr(phantom, r, data_range=d_range), ssim(phantom, r, data_range=d_range)))
        ax = axes[row, col]
        ax.imshow(r, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{f}, lam={lam}\nPSNR {rows[f][-1][1]:.1f} dB, SSIM {rows[f][-1][2]:.2f}")
        ax.set_xlabel("X (pixels)"); ax.set_ylabel("Y (pixels)")
plt.suptitle("Shepp-Logan: effect of the Poisson noise level on ramp (top) and Hann (bottom) FBP", y=1.0)
plt.tight_layout(); plt.show()

print(f"{'lam':>5} | {'noise std':>9} | {'ramp PSNR':>9} {'ramp SSIM':>9} | {'hann PSNR':>9} {'hann SSIM':>9}")
print("-" * 65)
for i, lam in enumerate(LAMS):
    print(f"{lam:>5} | {np.sqrt(lam):>9.2f} | {rows['ramp'][i][1]:>9.2f} {rows['ramp'][i][2]:>9.3f} | {rows['hann'][i][1]:>9.2f} {rows['hann'][i][2]:>9.3f}")


#### Discussion 1.5 – What, how and why

**What the Shepp–Logan phantom is.** It is a standard test image. Shepp and Logan made it in 1974 as a simple model of a human head. It is built from 10 ellipses [6]. It is harder than our two-lesion phantom: the skull is a thin bright ring (value 1), the brain inside has low contrast (0.2–0.3), and some small ellipses have a contrast of only 0.01–0.1.

**How we added the noise, and why.** Real CT noise is *quantum noise*. The number of X-ray photons that reach the detector is random, and it follows a **Poisson** distribution [15]. This is why the noise gets bigger when the dose gets smaller. As the task asks, we added `np.random.poisson(lam=10)` to every value of the clean sinogram. We never add noise to the image. In a real scanner the noise is in the measured projections, and the reconstruction then spreads it over the whole image. Poisson(10) has a mean of 10 and a standard deviation of √10 ≈ 3.2. The clean sinogram goes up to about 106, so the noise is only a few percent of the signal in the sinogram. But **FBP makes the noise much bigger**. We used 200 projections and a fixed random seed, so we get the same numbers every time we run the cell.

**Results (200 projections, λ = 10, profile at row y = 175):**

| filter | MSE | PSNR (dB) | SSIM | MAE |
|---|---|---|---|---|
| Unfiltered BP (rescaled) | 0.0361 | 14.43 | 0.351 | 0.1047 |
| ramp | 0.0186 | 17.30 | 0.276 | 0.0958 |
| shepp-logan | 0.0132 | 18.80 | 0.290 | 0.0800 |
| cosine | 0.0073 | 21.35 | 0.332 | 0.0573 |
| hamming | 0.0059 | 22.29 | 0.364 | 0.0493 |
| hann | 0.0056 | 22.51 | 0.377 | 0.0473 |

**Effect of the filters on the noisy reconstruction**
- *Ramp = sharp but very noisy.* The ramp image looks grainy. In the profile, the red line goes up and down by about ±0.3 around the true value. This is as big as the difference between brain and fluid in the phantom, so the small structures (the ventricles) are hard to see. Why? The ramp filter makes high frequencies stronger. In a noisy sinogram, the high frequencies are mostly noise. So the ramp filter makes the noise stronger too. A window (Hamming or Hann) is added to the ramp to stop this [16]. On a real CT scanner this is called the "kernel": a smooth kernel gives less noise but less detail, a sharp kernel gives more detail but more noise [17].
- *Softer filters = less noise.* Going from ramp → Shepp-Logan → cosine → Hamming → Hann, the noise gets smaller each time. MSE goes down from 0.0186 to 0.0056 (3 times better) and PSNR goes up by 5 dB. The brain looks smooth again, and the dark ellipses and the small shapes in the middle can be seen again.
- *The price is sharpness.* Soft filters also blur the skull ring a little (with Hann the peak is lower and wider). A sharp kernel = sharpest image but most noise. A smooth kernel = less noise but blurred details [17]. A radiologist makes the same choice on a real scanner (sharp kernel for bone, smooth kernel for soft tissue).
- *Unfiltered BP.* Still just a blur. After rescaling, its MSE (0.036) is 6 times worse than Hann. Its SSIM (0.35) is close to the filtered ones. This is because SSIM punishes the fine noise of FBP as much as the blur of BP. So one metric alone does not tell the whole story.

**What happens when we change the noise level λ? (cell "extra")**

| λ | noise std | ramp PSNR / SSIM | Hann PSNR / SSIM |
|---|---|---|---|
| 0 | 0 | 26.8 / 0.60 | 26.4 / 0.83 |
| 1 | 1.0 | 24.4 / 0.43 | 26.0 / 0.69 |
| 10 | 3.2 | 17.3 / 0.28 | 22.5 / 0.38 |
| 50 | 7.1 | 9.6 / 0.23 | 13.7 / 0.27 |
| 200 | 14.1 | 0.9 / 0.19 | 2.9 / 0.22 |

- A bigger λ means more noise (the noise size is √λ). So λ = 200 has 4.5 times more noise than λ = 10. At λ = 50 the ramp image is almost unreadable (9.6 dB), but Hann still shows the main shapes (13.7 dB). At λ = 200 even Hann does not work any more.
- Poisson noise also adds a constant value λ to the whole sinogram. This does not matter. The ramp filter has zero gain at frequency 0, so it removes this constant.
- Without noise, the ramp is a little better in PSNR (26.8 dB vs 26.4 dB). From λ = 1 the order changes, and at λ = 10 Hann is 5 dB better. So the "best" filter depends on the dose.
- More projections also help against noise, because each pixel is the average of more measurements (the noise goes down like 1/√n).

**Question – Differences between the two phantoms**
1. *Contrast matters.* In the simple phantom, every edge has a contrast of 1. So noise of ±0.1 was not visible, and all filters had SSIM above 0.67. The Shepp–Logan phantom has small contrasts of only 0.01–0.1 inside. This is smaller than the noise of the ramp reconstruction, so these shapes disappear if we do not smooth. SSIM falls to 0.28–0.38, because the fine noise breaks the local structure everywhere, even if the MSE is small.
2. *The filter becomes important.* Without noise (1.4), the five filters were almost the same (PSNR within about 1 dB). With noise (1.5), the filter changes the PSNR by 5 dB.
3. *A complex image needs more projections.* Shepp–Logan has many edges in many directions. With 100 projections we saw faint streaks, so we used 200.
4. *Same physics, same conclusion.* The ramp filter is the most exact on clean data. A windowed filter (Hann or Hamming) is the best on real, noisy data.

**Note on the code.** The unfiltered BP has its own scale (it is a sum of line integrals; the maximum is about 133 here). So before we compute the metrics, we fit it to the phantom with a straight line (`np.polyfit`). The filtered reconstructions are not rescaled, so their metrics can be compared directly between 1.4 and 1.5.e not rescaled, so their metrics are directly comparable between 1.4 and 1.5.

### References

1. DICOM Standard PS3.3, C.11.1 *Modality LUT Module* (Rescale Slope / Intercept → HU). https://dicom.nema.org/medical/dicom/2019e/output/chtml/part03/sect_C.11.html
2. Radiopaedia – *Hounsfield unit*. https://radiopaedia.org/articles/hounsfield-unit
3. CT windowing (window width / level): https://arxiv.org/pdf/2402.04267 (section 3); LITFL – *Abdominal CT: Windows*. https://litfl.com/abdominal-ct-windows-advanced/
4. Typical window settings (lung, bone, soft tissue). https://www.uomus.edu.iq/img/lectures21/MUCLecture_2025_21356500.pdf
5. Lorensen W.E., Cline H.E. (1987). *Marching cubes: a high resolution 3D surface construction algorithm.* Computer Graphics 21(4). https://courses.cs.duke.edu/cps124/fall01/resources/p163-lorensen.pdf
6. Shepp L.A., Logan B.F. (1974). *The Fourier reconstruction of a head section.* IEEE Trans. Nucl. Sci. – https://en.wikipedia.org/wiki/Shepp%E2%80%93Logan_phantom
7. scikit-image `radon` / `iradon` source and docs (FBP, Kak & Slaney). https://pytranskit.readthedocs.io/en/latest/_modules/skimage/transform/radon_transform.html
8. CT artifacts – undersampling / view aliasing. https://dlibra.umcs.lublin.pl/Content/44455/download
9. Kim G. et al. (2018). *Analytic CT reconstruction in sparse-angular sampling…* J. Korean Phys. Soc. 73, 361–367. https://link.springer.com/10.3938/jkps.73.361
10. Champley K.M. et al. (2024). *Methods for Few-View CT Image Reconstruction* (arXiv 2410.07552). https://arxiv.org/pdf/2410.07552
11. scikit-image gallery – *Radon transform*. https://scikit-image.org/docs/0.23.x/auto_examples/transform/plot_radon_transform.html
12. MathWorks – `iradon` filters (Ram-Lak, Shepp-Logan, Cosine, Hamming, Hann). https://www.mathworks.com/help/images/ref/iradon.html
13. MIT comp-photo notes – *Image metrics* (PSNR vs SSIM). https://people.csail.mit.edu/fredo/comp-photo-book/03-basic-image-processing-and-isp-10-image-metrics.html
14. Horé A., Ziou D. (2010). *Image quality metrics: PSNR vs. SSIM.* ICPR. https://projet.liris.cnrs.fr/imagine/pub/proceedings/ICPR-2010/data/4109c366.pdf ; Wang Z. et al. (2004). *Image Quality Assessment: From Error Visibility to Structural Similarity.* https://www.cns.nyu.edu/pub/lcv/wang03-reprint.pdf
15. Kachelriess M. (2003). *Noise prediction and simulation in CT.* RSNA. http://archive.rsna.org/2003/3100647.html ; mixed Poisson–Gaussian CT noise: https://arxiv.org/pdf/1801.09533
16. *Introduction of total variation regularization into filtered backprojection* (window on the ramp filter). https://arxiv.org/pdf/1711.00365
17. Yu L., Leng S. – *Image Reconstruction Techniques* (Image Wisely / ACR): smoother kernel = less noise but lower resolution, sharper kernel = more resolution but more noise. https://www.imagewisely.org/-/media/Image-Wisely/Files/CT/IW-Yu-Leng-Image-Reconstruction-Techniques.pdf
18. Kak A.C., Slaney M. (1988). *Principles of Computerized Tomographic Imaging*, IEEE Press – ch. 3 (FBP) and ch. 5 (Aliasing artifacts and noise in CT images). Free chapters: https://engineering.purdue.edu/~malcolm/pct/pct-toc.html
19. University of Cambridge, Engineering Tripos 3G4 crib (2025): derivation that the number of projections n ≈ π·m/2 for m detector samples. https://teaching.eng.cam.ac.uk/system/files/Crib_3G4_2025.pdf

###  Generative AI Declaration Policy
If generative AI tools were used in completing this practicum, you must explicitly declare their use. If you used AI, please include response the following three points:
1. ``Purpose``: A brief explanation of what you used generative AI for (e.g., brainstorming, proofreading, code debugging, or structuring ideas).
2. ``Sample Prompt``: At least one example of a prompt you submitted to the AI.
3. ``Refinement Process``: A short description of how you handled the prompt and evaluated the output (e.g., how you verified the facts, refined the wording, or altered the prompt to get better results).

## Generative AI Declaration

1. `Purpose`:
We used a generative AI assistant (Claude, Anthropic) to help write the discussion text
in simple English, to suggest the extra experiments (number of projections, angular range,
noise level lambda, filter responses), to find and check web references, and to review our code (it pointed out the phantom size, the repeated 0/360 degree projection and the wrong scale of the unfiltered BP; we applied the corrections ourselves).
2. `Sample prompt`:
"Can you use simple English, with discussion and comments for each question, using internet references for the why, how and what? Also pay attention to the professor's instructions about what happens when you change some values.
3. `Refinement process`:
All the code was written and run by us in Colab. We first wrote our own sample code for each section, and then asked the AI to make changes to improve how the plots and figures look (titles, labels, legends, layout). To choose the parameter values, we used ipywidgets sliders in Colab: we moved the sliders (for example the bone thresholds, the window level and width, the number of projections and the noise level) and watched how the results changed before we fixed the final values. Every number in the discussions comes from our own notebook outputs, and we ran the cells again to check them. We opened each reference and checked that it really supports the sentence where it is used. We made the AI text shorter and removed anything we could not check ourselves. We improved the answers step by step: we read each answer, made sure we understood it, added our own ideas and views, and compared the new answer with the old one to see what was missing and what was added. In this process we learned not only how to code, but also things we had not understood well before. When we did not understand something, we broke it into small parts and compared it with the lecture notes until it made sense, or until we found that the change made by the AI was wrong. The commits of the code can be found on github repository. Sometimes we had issues with syncing the latest changes from colab to git, so we had to replace the entire notebook with a new version, which may not result in a cleaner git history.

- https://github.com/sacumesh/medical-imagining-labs

### Group Organisation and Contribution

Since this assignment is a group project, your team must include a text addressing the following three points:
1. ``Task Organization``: Describe how your team organised the workflow to complete the entire notebook (e.g., who was responsible for which sections, how you collaborated, and how you merged your work).

2. ``Conflict Resolution``: Explain how you handled any discrepancies, disagreements, or differing opinions regarding the project's technical approach or final answers.

3. ``Percentage Contribution``: State the exact percentage of effort and contribution made by each member. The total sum across all three members must equal exactly 100% (e.g., Member A: 33.333%, Member B: 33.333%, Member C: 33.333%).
